# 🚀 Ornith-1.5-35B-A3B Server Launcher (2x Tesla T4)

### Pre-launch notes:
1. Right panel (Settings) -> **Accelerator**: select **GPU T4 x 2**
2. Right panel (Settings) -> **Internet**: set to **On**
3. *(optional)* For Cloudflare Named Tunnel (`api.zexnoz.dev`): **Add-ons** -> **Secrets** -> add `CF_TUNNEL_TOKEN` (if not set, a free Quick Tunnel is launched automatically)
4. Click **Run Cell 1** below to install and start the Server + Cloudflare Tunnel
5. Once done, you can run Cell 2 to chat, or use the Public URL from outside

In [ ]:
# ==============================================================================
# Kaggle / Colab T4 x2 -> Ornith-1.5-35B-A3B + Cloudflare Tunnel (api.zexnoz.dev)
# ==============================================================================
import argparse
import json
import os
import re
import shutil
import subprocess
import sys
import tarfile
import time
import urllib.request
from pathlib import Path

# ================= Configuration =================
DEFAULT_MODEL_REPO = "gbuzhf/Ornith-1.5-35B-A3B-TIEL-Calibrated-MTPv2-ICE-GGUF"
DEFAULT_MODEL_FILE = "Ornith-1.5-35B-A3B-TIEL_Calibrated-MTPv2-21G-ICE.gguf"
MODEL_ALIAS = "ornith-1.5-35B-A3B"
STATIC_DOMAIN = "https://api.zexnoz.dev"
CONTEXT_SIZE = 196608        # 192k tokens
BATCH_SIZE = 1024            # 1024 for VRAM stability in long context
UBATCH_SIZE = 512
KV_CACHE_TYPE = "q4_0"
SERVER_PORT = 8080
API_KEY = "kilo-secret-key"
# =================================================

WORKDIR = Path("/kaggle/tmp/llm_server" if Path("/kaggle").exists() else "/tmp/llm_server")
BIN_DIR = WORKDIR / "bin"
MODEL_DIR = Path("/kaggle/tmp/models" if Path("/kaggle").exists() else "/tmp/models")
LOG_DIR = Path("/kaggle/working" if Path("/kaggle").exists() else WORKDIR / "logs")
CLOUDFLARED_BIN = WORKDIR / "cloudflared"
LLAMA_LOG = LOG_DIR / "llama-server.log"
CF_LOG = LOG_DIR / "cloudflared.log"
USER_AGENT = "kaggle-llm-installer/1.0"

def stop_services():
    print("🧹 Cleaning up existing processes...")
    os.system("pkill -9 -f '[l]lama-server' >/dev/null 2>&1")
    os.system("pkill -9 -f '[c]loudflared' >/dev/null 2>&1")
    time.sleep(1)
    print("✅ Cleanup complete.")

stop_services()

def run_cmd(cmd, check=True, capture=False, env=None):
    print(f"$ {' '.join(str(x) for x in cmd)}", flush=True)
    return subprocess.run(
        [str(x) for x in cmd],
        check=check,
        text=True,
        stdout=subprocess.PIPE if capture else None,
        stderr=subprocess.STDOUT if capture else None,
        env=env,
    )

def setup_directories():
    WORKDIR.mkdir(parents=True, exist_ok=True)
    BIN_DIR.mkdir(parents=True, exist_ok=True)
    MODEL_DIR.mkdir(parents=True, exist_ok=True)
    LOG_DIR.mkdir(parents=True, exist_ok=True)

def check_gpu():
    if not shutil.which("nvidia-smi"):
        raise RuntimeError("nvidia-smi not found. Enable GPU accelerator.")
    out = run_cmd(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"], capture=True).stdout
    print(f"GPUs detected:\n{out.strip()}")

def get_cf_token():
    """Get token from Kaggle Secrets or Environment Variable"""
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret("CF_TUNNEL_TOKEN")
    except Exception:
        return os.environ.get("CF_TUNNEL_TOKEN", "")

def install_prebuilt_llamacpp():
    server_bin = BIN_DIR / "llama-server"
    if server_bin.exists():
        print("✅ Found existing llama-server binary.")
        return server_bin
    
    # Search existing
    found = [p for p in WORKDIR.rglob("llama-server") if p.is_file() and os.access(p, os.X_OK)]
    if found:
        try:
            server_bin.symlink_to(found[0])
        except Exception:
            shutil.copy2(found[0], server_bin)
        return server_bin

    print("\n[1/4] Downloading prebuilt llama.cpp CUDA...")
    api_url = "https://api.github.com/repos/cloudlnkcn/llama.cpp/releases?per_page=5"
    req = urllib.request.Request(api_url, headers={"User-Agent": USER_AGENT})
    
    download_url = None
    archive_name = "ubuntu-cuda-sm_75-x64.tar.xz"
    try:
        with urllib.request.urlopen(req, timeout=15) as resp:
            releases = json.loads(resp.read().decode())
            pattern = re.compile(r"ubuntu-cuda-sm_75-x64\.tar\.xz$", re.I)
            for rel in releases:
                for asset in rel.get("assets", []):
                    if pattern.search(asset["name"]):
                        download_url = asset["browser_download_url"]
                        archive_name = asset["name"]
                        break
                if download_url:
                    break
    except Exception as e:
        print(f"⚠️ Github API warning: {e}, using fallback...")
    
    if not download_url:
        download_url = "https://github.com/ai-dock/llama.cpp-cuda/releases/download/b9628/llama.cpp-b9628-cuda-12.8-amd64.tar.gz"
        archive_name = "llama.cpp-b9628-cuda-12.8-amd64.tar.gz"
    
    archive_path = WORKDIR / archive_name
    print(f"Downloading: {download_url}")
    urllib.request.urlretrieve(download_url, archive_path)
    print("Extracting llama.cpp...")
    with tarfile.open(archive_path, "r:*") as tf:
        tf.extractall(WORKDIR)
    
    found = list(WORKDIR.rglob("llama-server"))
    if not found:
        raise RuntimeError("llama-server executable not found.")
    
    real_server = found[0]
    real_server.chmod(real_server.stat().st_mode | 0o755)
    
    if server_bin.exists():
        server_bin.unlink()
    try:
        server_bin.symlink_to(real_server)
    except Exception:
        shutil.copy2(real_server, server_bin)
    print(f"✅ llama-server ready: {server_bin}")
    return server_bin

def install_cloudflared():
    if CLOUDFLARED_BIN.exists():
        return CLOUDFLARED_BIN
    print("\n[2/4] Downloading Cloudflare Tunnel (cloudflared)...")
    url = "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64"
    urllib.request.urlretrieve(url, CLOUDFLARED_BIN)
    CLOUDFLARED_BIN.chmod(0o755)
    print(f"✅ cloudflared ready: {CLOUDFLARED_BIN}")
    return CLOUDFLARED_BIN

def download_model(repo_id: str, filename: str) -> Path:
    # Check mounted dataset
    if Path("/kaggle/input").exists():
        mounted = [p for p in Path("/kaggle/input").rglob("*.gguf") if "mmproj" not in p.name.lower()]
        if mounted:
            print(f"⚡ Found mounted dataset: {mounted[0]} ({mounted[0].stat().st_size / (1024**3):.2f} GB)")
            return mounted[0]
    
    target = MODEL_DIR / filename
    if target.exists() and target.stat().st_size > 1024 * 1024 * 1024:
        print(f"✅ Found existing model: {target} ({target.stat().st_size / (1024**3):.2f} GB)")
        return target
    
    print(f"\n[3/4] Downloading GGUF model: {repo_id}/{filename}...")
    try:
        from huggingface_hub import hf_hub_download
        downloaded = hf_hub_download(
            repo_id=repo_id,
            filename=filename,
            local_dir=str(MODEL_DIR),
        )
        return Path(downloaded)
    except Exception:
        url = f"https://huggingface.co/{repo_id}/resolve/main/{filename}"
        if shutil.which("aria2c"):
            run_cmd(["aria2c", "-x", "16", "-s", "16", "-k", "1M", "-d", str(MODEL_DIR), "-o", filename, url])
        else:
            run_cmd(["wget", "--continue", "-O", str(target), url])
        return target

def wait_for_server_ready(port: int, timeout_sec: int = 240):
    print("⏳ Waiting for llama-server to load model into VRAM and initialize...")
    start_t = time.time()
    while time.time() - start_t < timeout_sec:
        try:
            req = urllib.request.Request(
                f"http://127.0.0.1:{port}/v1/models",
                headers={"Authorization": f"Bearer {API_KEY}"} if API_KEY else {}
            )
            with urllib.request.urlopen(req, timeout=3) as resp:
                if resp.status == 200:
                    print("✅ llama-server is ready and responding!")
                    return True
        except Exception:
            pass
        time.sleep(2)
    return False

def start_server(server_bin: Path, model_path: Path, context_size: int):
    print(f"\n[4/4] Starting llama-server & Cloudflare Tunnel...")
    
    cf_token = get_cf_token()
    so_dirs = {str(p.parent.resolve()) for p in WORKDIR.rglob("*.so*") if p.is_file()}
    ld_path = ":".join(so_dirs) + ":" + os.environ.get("LD_LIBRARY_PATH", "")
    env = os.environ.copy()
    env["LD_LIBRARY_PATH"] = ld_path
    env["CUDA_VISIBLE_DEVICES"] = "0,1"
    env["GGML_CUDA_NO_VMM"] = "1"
    
    help_text = subprocess.run([str(server_bin), "--help"], capture_output=True, text=True, env=env).stdout or ""
    
    server_cmd = [
        str(server_bin),
        "-m", str(model_path),
        "-ngl", "99",
        "-sm", "layer",
        "-c", str(context_size),
        "-b", str(BATCH_SIZE),
        "-ub", str(UBATCH_SIZE),
        "-np", "1",
        "--cache-type-k", KV_CACHE_TYPE,
        "--cache-type-v", KV_CACHE_TYPE,
        "--tensor-split", "1,1",
        "--host", "127.0.0.1",
        "--port", str(SERVER_PORT),
        "--api-key", API_KEY,
        "--alias", MODEL_ALIAS,
    ]
    if "--spec-type" in help_text:
        server_cmd.extend(["--spec-type", "draft-mtp"])
    if "--flash-attn" in help_text:
        server_cmd.extend(["--flash-attn", "on"])
    elif "-fa" in help_text:
        server_cmd.extend(["-fa", "on"])
    
    server_log = open(LLAMA_LOG, "w", encoding="utf-8", buffering=1)
    server_proc = subprocess.Popen(
        server_cmd,
        stdout=server_log,
        stderr=subprocess.STDOUT,
        env=env,
        text=True,
        start_new_session=True
    )
    
    # Start Cloudflare Tunnel (Named Tunnel or Quick Tunnel fallback)
    cf_log = open(CF_LOG, "w", encoding="utf-8", buffering=1)
    public_url = None
    if cf_token:
        print(f"🔐 Launching Cloudflare Named Tunnel ({STATIC_DOMAIN})...")
        cf_proc = subprocess.Popen(
            [str(CLOUDFLARED_BIN), "tunnel", "run", "--token", cf_token],
            stdout=cf_log,
            stderr=subprocess.STDOUT,
            text=True,
            start_new_session=True
        )
        public_url = STATIC_DOMAIN
    else:
        print("⚠️ No CF_TUNNEL_TOKEN found. Launching free Quick Tunnel fallback...")
        cf_proc = subprocess.Popen(
            [str(CLOUDFLARED_BIN), "tunnel", "--url", f"http://127.0.0.1:{SERVER_PORT}"],
            stdout=cf_log,
            stderr=subprocess.STDOUT,
            text=True,
            start_new_session=True
        )
        start_t = time.time()
        while time.time() - start_t < 45:
            if CF_LOG.exists():
                match = re.search(r"https://[-0-9a-z]+\.trycloudflare\.com", CF_LOG.read_text(errors="replace"))
                if match:
                    public_url = match.group(0)
                    break
            time.sleep(2)
    
    # Wait for server
    if not wait_for_server_ready(SERVER_PORT):
        print("⚠️ llama-server timeout. Check logs:")
        print((LLAMA_LOG).read_text()[-1500:])
        return
    
    print("\n" + "=" * 74)
    print("🚀 ORNITH-1.5-35B-A3B SERVER ONLINE (2x TESLA T4)")
    print("=" * 74)
    print(f"📡 Base URL       : {public_url}/v1" if public_url else f"🖥️ Local URL: http://127.0.0.1:{SERVER_PORT}/v1")
    print(f"🔑 API Key        : {API_KEY}")
    print(f"🤖 Model ID       : {MODEL_ALIAS}")
    print(f"🧠 Context Length : {context_size:,} tokens (128k)")
    print(f"⚡ Batch / UBatch : {BATCH_SIZE} / {UBATCH_SIZE} (KV: {KV_CACHE_TYPE})")
    print("=" * 74)
    print("\n✅ Server is running in background. You can run the next cell to test right away")

setup_directories()
check_gpu()
server_bin = install_prebuilt_llamacpp()
install_cloudflared()
DEFAULT_MODEL_REPO = "gbuzhf/Ornith-1.5-35B-A3B-TIEL-Calibrated-MTPv2-ICE-GGUF"
start_server(server_bin, model_path, CONTEXT_SIZE)


In [ ]:
# ==============================================================================
# 💬 Cell 2: Test chat with Ornith-1.5-35B-A3B (ask a question right in the notebook)
# ==============================================================================
import json
import urllib.request

def ask_ornith(prompt, max_tokens=512):
    url = 'http://127.0.0.1:8080/v1/chat/completions'
    payload = {
        'model': 'ornith-1.5-35B-A3B',
        'messages': [{'role': 'user', 'content': prompt}],
        'max_tokens': max_tokens,
        'temperature': 0.7
    }
    req = urllib.request.Request(
        url,
        data=json.dumps(payload).encode('utf-8'),
        headers={
            'Content-Type': 'application/json',
            'Authorization': 'Bearer kilo-secret-key'
        }
    )
    with urllib.request.urlopen(req, timeout=120) as resp:
        res = json.loads(resp.read().decode('utf-8'))
        return res['choices'][0]['message']['content']

# Test question
prompt = 'Hello, briefly introduce yourself, and tell me what model you are and what you can do'
print(f'👤 User: {prompt}\n')
print('🤖 Ornith is thinking...')
reply = ask_ornith(prompt)
print(f'\n{reply}')


In [ ]:
# ==============================================================================
# 🛑 Cell 3: Stop all servers (run when you want to stop)
# ==============================================================================
stop_services()
